<a href="https://colab.research.google.com/github/yoshiagent/mathModel-colab/blob/main/notebooks/InvertPole_Suspension.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="在 Colab 開啟"/></a>

# 倒單擺與汽車懸吊系統

對照網頁：[倒單擺](https://yoshiagent.github.io/mathModel/IP.html)、[汽車懸吊系統](https://yoshiagent.github.io/mathModel/CS.html)

| 原檔／原講義 | 內容 | 本筆記本 |
|---|---|---|
| 倒單擺推導 (1)～(7) | 用 Lagrange 方程式重新推導，和網頁的 (6)、(7) 式比對 | 第 1 節 |
| `InvertPole.zip`：`slcp.mdl` 的 Cart & Pole Dynamics | Fcn1、Fcn2 兩個 f(u) 方塊、桿長 1 m（$l$ = 0.5）、桿 0.1 kg、台車 1 kg | 第 2 節 |
| `slcp.fis`、`initcp.m` | 16 條規則的 Sugeno 模糊控制器，追蹤 ±1.5 m 方波目標 | 第 3 節 |
| 線性化 | 直立點的極點：為什麼一定要控制 | 第 4 節 |
| 汽車懸吊（半車模型） | 由四條運動方程式推出 $\mathbf M$、$\mathbf C$、$\mathbf K$、$\mathbf D$、$\mathbf E$，驗證網頁的更正；被動懸吊過路面凸塊 | 第 5 節 |

只用 Colab 內建的 numpy、scipy、sympy、plotly。

In [1]:
import numpy as np
import sympy as sp
from scipy.integrate import solve_ivp
import plotly.graph_objects as go
from plotly.subplots import make_subplots


def show(fig, title, xlab=None, ylab=None, h=420):
    fig.update_layout(title=title, xaxis_title=xlab, yaxis_title=ylab, height=h, template="plotly_white",
                      margin=dict(l=60, r=20, t=50, b=50))
    fig.show()

## 1. 用 Lagrange 方程式驗證倒單擺的運動方程式

座標同網頁：台車水平位移 $y$、桿與鉛直線夾角 $\theta$；桿長 $2l$、質量 $m$、對質心的慣量 $I = \frac{1}{12}m(2l)^2$；台車質量 $M$、水平力 $u$。
桿的質心 $G$：水平 $y + l\sin\theta$、鉛直 $l\cos\theta$。

In [2]:
t = sp.symbols("t")
m, M, l, g, u = sp.symbols("m M l g u", positive=True)
y, th = sp.Function("y")(t), sp.Function("theta")(t)
I = sp.Rational(1, 12) * m * (2 * l) ** 2
xG, yG = y + l * sp.sin(th), l * sp.cos(th)
T = sp.Rational(1, 2) * M * y.diff(t) ** 2 + sp.Rational(1, 2) * m * (xG.diff(t) ** 2 + yG.diff(t) ** 2) + sp.Rational(1, 2) * I * th.diff(t) ** 2
V = m * g * yG
Lg = T - V
eq_y = sp.simplify(Lg.diff(y.diff(t)).diff(t) - Lg.diff(y) - u)
eq_th = sp.simplify(Lg.diff(th.diff(t)).diff(t) - Lg.diff(th))
ydd, thdd = sp.symbols("yddot thetaddot")
rep = {y.diff(t, 2): ydd, th.diff(t, 2): thdd}
sol = sp.solve([eq_y.subs(rep), eq_th.subs(rep)], [ydd, thdd], dict=True)[0]

thd = th.diff(t)
web6 = (g * sp.sin(th) + sp.cos(th) * (-u - m * l * thd ** 2 * sp.sin(th)) / (M + m)) / (l * (sp.Rational(4, 3) - m / (M + m) * sp.cos(th) ** 2))
web7 = (u + m * l * thd ** 2 * sp.sin(th) - m * l * thdd * sp.cos(th)) / (M + m)
print("Lagrange theta'' - eq.(6) =", sp.simplify(sol[thdd] - web6))
print("Lagrange y''     - eq.(7) =", sp.simplify((sol[ydd] - web7.subs(thdd, sol[thdd]))))
print("eq.(4):", sp.simplify(eq_th.subs(rep) / l), "= 0   (cf. (m l cos th) y'' + (I + m l^2) th'' = m g l sin th)")

Lagrange theta'' - eq.(6) = 0
Lagrange y''     - eq.(7) = 0
eq.(4): m*(-3*g*sin(theta(t)) + 4*l*thetaddot + 3*yddot*cos(theta(t)))/3 = 0   (cf. (m l cos th) y'' + (I + m l^2) th'' = m g l sin th)


## 2. Simulink 範例的倒單擺模型（`slcp.mdl`）

Cart & Pole Dynamics 方塊（遮罩參數 Physical Specifications = `[1.0, 0.1, 1.0, 9.8]`，初始化 `l=specs(1)/2; m=specs(2); mc=specs(3); g=specs(4);`）
內部就是網頁的 (6)、(7) 式：
```
Fcn1: (g*u(2)+u(3)*((-u(1)-m*l*u(4)*u(4)*u(2))/(mc+m)))/(l*(4/3-(m*u(3)*u(3))/(mc+m)))   % θ''
Fcn2: (u(1)+m*l*(u(4)*u(4)*u(2)-u(5)*u(3)))/(mc+m)                                        % y''
```
先看不加控制時：桿子從 $\theta_0$ = 0.01 rad 開始倒下。

In [3]:
SPEC = dict(l=1.0 / 2, m=0.1, mc=1.0, g=9.8)


def cartpole(state, F, p=SPEC):
    """state = (theta, theta_dot, y, y_dot)，照 Fcn1、Fcn2 的寫法"""
    th_, thd_, y_, yd_ = state
    l_, m_, mc, g_ = p["l"], p["m"], p["mc"], p["g"]
    s, c = np.sin(th_), np.cos(th_)
    thdd_ = (g_ * s + c * ((-F - m_ * l_ * thd_ * thd_ * s) / (mc + m_))) / (l_ * (4 / 3 - (m_ * c * c) / (mc + m_)))
    ydd_ = (F + m_ * l_ * (thd_ * thd_ * s - thdd_ * c)) / (mc + m_)
    return np.array([thd_, thdd_, yd_, ydd_])


solf = solve_ivp(lambda tt, x: cartpole(x, 0.0), (0, 2.5), [0.01, 0, 0, 0], t_eval=np.linspace(0, 2.5, 251), rtol=1e-9, atol=1e-11)
k90 = np.argmax(np.abs(solf.y[0]) > np.pi / 2)
print(f"without control the pole reaches 90 deg at t = {solf.t[k90]:.2f} s; cart moves {solf.y[2, k90]:.3f} m")
fig = go.Figure()
fig.add_scatter(x=solf.t, y=np.round(np.degrees(solf.y[0]), 3), name="theta (deg)")
show(fig, "Open loop: the inverted pendulum falls (theta0 = 0.01 rad)", "t (s)", "theta (deg)")

without control the pole reaches 90 deg at t = 1.47 s; cart moves -0.045 m


## 3. 模糊控制器 `slcp.fis`（Sugeno，16 條規則）

輸入：$\theta$（±0.3 rad）、$\dot\theta$（±1）、$y - y_{target}$（±3 m）、$\dot y$（±3）；每個輸入兩個 gbell 隸屬函數；
規則的結論是一次式 $u = a_1\theta + a_2\dot\theta + a_3(y - y_t) + a_4\dot y + a_0$，以 prod 計算激發強度、加權平均解模糊化。
目標位置：方波，振幅 1.5 m、0.62832 rad/s（週期 10 s）；`slcp.mdl` 用 ode5 固定步長 0.01 s。

In [4]:
FIS_OUT = np.array([
    [41.3729301501, 10.029755738, 3.1622770253, 4.2875458985, 0.3385581576], [40.4085343053, 10.0534292791, 3.1622772529, 4.2875459027, 0.2068296814],
    [41.3729288678, 10.0297560085, 3.1622772519, 4.2875457869, 0.3385606013], [40.4085347194, 10.053429123, 3.1622772565, 4.2875457708, 0.2068326351],
    [38.5608910694, 10.1773506475, 3.1622772869, 4.2875456991, -0.0489283614], [37.5964961463, 10.1536776552, 3.1622772208, 4.2875456535, -0.180656185],
    [38.5608914913, 10.1773508119, 3.1622772852, 4.2875457216, -0.0489261838], [37.5964964971, 10.1536775955, 3.1622771613, 4.2875456734, -0.180653468],
    [37.5963645972, 10.1536683258, 3.1622773123, 4.2875482313, 0.1806554998], [38.5608129606, 10.1773413565, 3.1622772085, 4.2875482844, 0.048912347],
    [37.5963624546, 10.1536681579, 3.1622771754, 4.2875481685, 0.1806581405], [38.5608148583, 10.1773416641, 3.162277231, 4.2875481347, 0.0489150654],
    [40.4083374749, 10.0534205134, 3.1622772797, 4.2875483525, -0.2068123866], [41.3727899363, 10.0297470643, 3.1622771749, 4.2875483317, -0.3385566739],
    [40.4083389119, 10.0534200783, 3.162277292, 4.2875484208, -0.2068100332], [41.3727895785, 10.0297473694, 3.1622771911, 4.2875484286, -0.33855436]])
MF = [[(0.3, 2, -0.3), (0.3, 2, 0.3)], [(1, 2, -1), (1, 2, 1)], [(3, 2, -3), (3, 2, 3)], [(3, 2, -3), (3, 2, 3)]]   # gbellmf [a b c]
RULES = [(i, j, k, n) for i in (0, 1) for j in (0, 1) for k in (0, 1) for n in (0, 1)]                            # 規則順序同 .fis


def gbell(x, a, b, c):
    return 1.0 / (1.0 + np.abs((x - c) / a) ** (2 * b))


def fuzzy_u(x):
    mu = [[gbell(x[i], *MF[i][0]), gbell(x[i], *MF[i][1])] for i in range(4)]
    w = np.array([mu[0][a] * mu[1][b] * mu[2][c] * mu[3][d] for a, b, c, d in RULES])
    z = FIS_OUT[:, :4] @ x + FIS_OUT[:, 4]
    return float(w @ z / w.sum())


def target(tt):
    return 1.5 * (1.0 if np.sin(0.62832 * tt) >= 0 else -1.0)


dt, tf = 0.01, 20.0
n_steps = int(round(tf / dt))
X = np.zeros((n_steps + 1, 4))
U = np.zeros(n_steps + 1)
TG = np.zeros(n_steps + 1)
for k in range(n_steps + 1):                                         # ode5 換成 RK4（同樣固定步長 0.01 s），控制量每步取樣一次
    tt = k * dt
    TG[k] = target(tt)
    U[k] = fuzzy_u(np.array([X[k, 0], X[k, 1], X[k, 2] - TG[k], X[k, 3]]))
    if k == n_steps:
        break
    f = lambda s: cartpole(s, U[k])
    k1 = f(X[k]); k2 = f(X[k] + dt / 2 * k1); k3 = f(X[k] + dt / 2 * k2); k4 = f(X[k] + dt * k3)
    X[k + 1] = X[k] + dt / 6 * (k1 + 2 * k2 + 2 * k3 + k4)
tt = np.arange(n_steps + 1) * dt
print(f"max |theta| = {np.degrees(np.abs(X[:, 0]).max()):.2f} deg, max |u| = {np.abs(U).max():.2f} N")
for t_chk in (4.99, 9.99, 14.99, 19.99):
    i = int(round(t_chk / dt))
    print(f"t = {t_chk:5.2f} s: target {TG[i]:+.1f} m, cart {X[i, 2]:+.4f} m, theta {np.degrees(X[i, 0]):+.4f} deg")
j = int(round(5.0 / dt))                                              # t = 5 s 目標由 +1.5 跳到 -1.5
print(f"after the target jumps to -1.5 m at t = 5 s the cart first moves to {X[j:j + 100, 2].max():+.4f} m (opposite direction), "
      f"then heads to the target;  steady offset at t = 19.99 s: {X[-2, 2] - TG[-2]:+.4f} m (from the rule constants a0)")
print("fuzzy u at (theta=0.1, others 0) =", round(fuzzy_u(np.array([0.1, 0, 0, 0])), 4), " ~ linear gain", round(FIS_OUT[:, 0].mean() * 0.1, 4))
fig = make_subplots(rows=3, cols=1, shared_xaxes=True, subplot_titles=("cart position y (m)", "pole angle theta (deg)", "force u (N)"))
fig.add_scatter(x=tt[::2], y=np.round(TG[::2], 3), name="target", line=dict(dash="dot"), row=1, col=1)
fig.add_scatter(x=tt[::2], y=np.round(X[::2, 2], 4), name="cart", row=1, col=1)
fig.add_scatter(x=tt[::2], y=np.round(np.degrees(X[::2, 0]), 4), name="theta", row=2, col=1)
fig.add_scatter(x=tt[::2], y=np.round(U[::2], 4), name="u", row=3, col=1)
fig.update_xaxes(title="t (s)", row=3, col=1)
show(fig, "slcp.mdl: Sugeno fuzzy controller keeps the pole up while the cart follows +/-1.5 m", h=700)

max |theta| = 16.15 deg, max |u| = 9.54 N
t =  4.99 s: target +1.5 m, cart +1.5046 m, theta +0.1343 deg
t =  9.99 s: target -1.5 m, cart -1.5077 m, theta -0.2518 deg
t = 14.99 s: target +1.5 m, cart +1.5077 m, theta +0.2517 deg
t = 19.99 s: target -1.5 m, cart -1.5077 m, theta -0.2517 deg
after the target jumps to -1.5 m at t = 5 s the cart first moves to +1.6238 m (opposite direction), then heads to the target;  steady offset at t = 19.99 s: -0.0077 m (from the rule constants a0)
fuzzy u at (theta=0.1, others 0) = 3.9048  ~ linear gain 3.9485


目標往左跳時，台車會先往**右**多走一點，讓桿子先向左傾，再整個移向目標（倒單擺的非最小相位特性）。
停下來的位置和目標差約 8 mm、桿子偏 0.25°，來自規則結論的常數項 $a_0$。

## 4. 線性化：直立點不穩定

在 $\theta = 0$、$\dot\theta = 0$ 附近，$\sin\theta \approx \theta$、$\cos\theta \approx 1$、忽略 $\dot\theta^2$：
$$\ddot\theta \approx \frac{g\theta - \frac{u}{M+m}}{l\left(\frac{4}{3} - \frac{m}{M+m}\right)}$$
特徵根 $\pm\sqrt{g/\left(l\left(\frac43 - \frac{m}{M+m}\right)\right)}$ 有一個在右半平面，所以不加控制必然倒下。

In [5]:
eps = 1e-6
A = np.zeros((4, 4))
for j in range(4):
    d = np.zeros(4); d[j] = eps
    A[:, j] = (cartpole(d, 0.0) - cartpole(-d, 0.0)) / (2 * eps)
Bv = (cartpole(np.zeros(4), eps) - cartpole(np.zeros(4), -eps)) / (2 * eps)
a = np.sqrt(SPEC["g"] / (SPEC["l"] * (4 / 3 - SPEC["m"] / (SPEC["mc"] + SPEC["m"]))))
print("A =\n", np.round(A, 4), "\nB =", np.round(Bv, 4))
print("open-loop eigenvalues:", np.round(np.sort(np.linalg.eigvals(A).real), 4), f"  (formula +/-{a:.4f})")
Kf = FIS_OUT[:, :4].mean(axis=0)
Acl = A + np.outer(Bv, Kf)                                          # 控制律 u = +K x（係數為正）
print("closed loop with the average fuzzy gains", np.round(Kf, 3), "-> eigenvalues", np.round(np.linalg.eigvals(Acl), 3))

A =
 [[ 0.      1.      0.      0.    ]
 [15.7756  0.      0.      0.    ]
 [ 0.      0.      0.      1.    ]
 [-0.7171  0.      0.      0.    ]] 
B = [ 0.     -1.4634  0.      0.9756]
open-loop eigenvalues: [-3.9719  0.      0.      3.9719]   (formula +/-3.9719)
closed loop with the average fuzzy gains [39.485 10.104  3.162  4.288] -> eigenvalues [-1.154+1.147j -1.154-1.147j -4.408+0.j    -3.887+0.j   ]


## 5. 汽車懸吊：半車模型

網頁的四條運動方程式（彈簧力、阻尼力在工作點線性化：$f_{ka} = k_{1a}(x_{1a} - x_{2a})$、$f_{ca} = c_a(\dot x_{1a} - \dot x_{2a})$，後輪同理），
加上剛體拘束 $x_c = \frac{l_b x_{1a} + l_a x_{1b}}{l}$、$\theta = \frac{x_{1a} - x_{1b}}{l}$。
用 sympy 整理成 $\mathbf M\ddot{\mathbf x} + \mathbf C\dot{\mathbf x} + \mathbf K\mathbf x = \mathbf D\mathbf f + \mathbf E\mathbf w$，
$\mathbf x = (x_{1a}, x_{2a}, x_{1b}, x_{2b})^T$、$\mathbf w = (\dot w_a, w_a, \dot w_b, w_b)^T$。

In [6]:
tt_ = sp.symbols("t")
mc_, Ic, la, lb, m2a, m2b, ca, cb, k1a, k1b, k2a, k2b = sp.symbols("m_c I_c l_a l_b m_2a m_2b c_a c_b k_1a k_1b k_2a k_2b", positive=True)
x1a, x2a, x1b, x2b, wa, wb = [sp.Function(nm)(tt_) for nm in ("x1a", "x2a", "x1b", "x2b", "w_a", "w_b")]
fa, fb = sp.symbols("f_a f_b")
L_ = la + lb
xc = (lb * x1a + la * x1b) / L_
theta = (x1a - x1b) / L_
fca, fka = ca * (x1a.diff(tt_) - x2a.diff(tt_)), k1a * (x1a - x2a)
fcb, fkb = cb * (x1b.diff(tt_) - x2b.diff(tt_)), k1b * (x1b - x2b)
eqs = [mc_ * xc.diff(tt_, 2) + fca + fka + fcb + fkb - fa - fb,
       Ic * theta.diff(tt_, 2) + la * (fca + fka - fa) - lb * (fcb + fkb - fb),
       m2a * x2a.diff(tt_, 2) - fca - fka + k2a * (x2a - wa) + fa,
       m2b * x2b.diff(tt_, 2) - fcb - fkb + k2b * (x2b - wb) + fb]
q = [x1a, x2a, x1b, x2b]
wv = [wa.diff(tt_), wa, wb.diff(tt_), wb]
Mm = sp.Matrix([[sp.expand(e).coeff(v.diff(tt_, 2)) for v in q] for e in eqs])
Cm = sp.Matrix([[sp.expand(e).coeff(v.diff(tt_)) for v in q] for e in eqs])
Km = sp.Matrix([[sp.expand(e).subs({v.diff(tt_, k): 0 for v in q for k in (1, 2)}).coeff(v) for v in q] for e in eqs])
Dm = sp.Matrix([[-sp.expand(e).coeff(f) for f in (fa, fb)] for e in eqs])
Em = sp.Matrix([[-sp.expand(e).coeff(w_) for w_ in wv] for e in eqs])
for name, Mat in (("M", Mm), ("C", Cm), ("K", Km), ("D", Dm), ("E", Em)):
    print(f"{name} =", sp.simplify(Mat).tolist())

M = [[l_b*m_c/(l_a + l_b), 0, l_a*m_c/(l_a + l_b), 0], [I_c/(l_a + l_b), 0, -I_c/(l_a + l_b), 0], [0, m_2a, 0, 0], [0, 0, 0, m_2b]]
C = [[c_a, -c_a, c_b, -c_b], [c_a*l_a, -c_a*l_a, -c_b*l_b, c_b*l_b], [-c_a, c_a, 0, 0], [0, 0, -c_b, c_b]]
K = [[k_1a, -k_1a, k_1b, -k_1b], [k_1a*l_a, -k_1a*l_a, -k_1b*l_b, k_1b*l_b], [-k_1a, k_1a + k_2a, 0, 0], [0, 0, -k_1b, k_1b + k_2b]]
D = [[1, 1], [l_a, -l_b], [-1, 0], [0, -1]]
E = [[0, 0, 0, 0], [0, 0, 0, 0], [0, k_2a, 0, 0], [0, 0, 0, k_2b]]


結果與網頁一致；$\mathbf K$ 的左上角是 $k_{1a}$（原圖寫成 $k_1$），$\mathbf E$ 的 $k_{2a}$、$k_{2b}$ 在第 2、4 欄（乘 $w_a$、$w_b$）。

**被動懸吊（$f_a = f_b = 0$）過路面凸塊**：網頁沒有給數值，這裡用一組**假設的**半車參數示範
（半個車體 600 kg、俯仰慣量 800 kg·m²、$l_a$ = 1.2 m、$l_b$ = 1.4 m、輪 40 kg、懸吊彈簧 20 kN/m、阻尼 1.5 kN·s/m、輪胎 200 kN/m），
車速 10 m/s 通過高 5 cm、長 0.5 m 的凸塊，後輪晚 $l/v$ = 0.26 s 碰到。

In [7]:
par = {mc_: 600, Ic: 800, la: 1.2, lb: 1.4, m2a: 40, m2b: 40, ca: 1500, cb: 1500, k1a: 2e4, k1b: 2e4, k2a: 2e5, k2b: 2e5}
Mn, Cn, Kn, En = (np.array(Mat.subs(par), dtype=float) for Mat in (Mm, Cm, Km, Em))
v, h, Lb = 10.0, 0.05, 0.5
lag = 2.6 / v


def bump(tt0):
    if 0 <= tt0 <= Lb / v:
        return h / 2 * (1 - np.cos(2 * np.pi * v * tt0 / Lb)), h / 2 * 2 * np.pi * v / Lb * np.sin(2 * np.pi * v * tt0 / Lb)
    return 0.0, 0.0


def road(tt0):
    wa_, dwa = bump(tt0 - 0.1)
    wb_, dwb = bump(tt0 - 0.1 - lag)
    return np.array([dwa, wa_, dwb, wb_])


Minv = np.linalg.inv(Mn)
rhs = lambda tt0, s: np.concatenate([s[4:], Minv @ (En @ road(tt0) - Cn @ s[4:] - Kn @ s[:4])])
ts = np.linspace(0, 3, 3001)
sol5 = solve_ivp(rhs, (0, 3), np.zeros(8), t_eval=ts, max_step=0.002, rtol=1e-8, atol=1e-10)
x1a_, x2a_, x1b_, x2b_ = sol5.y[:4]
xc_ = (1.4 * x1a_ + 1.2 * x1b_) / 2.6
th_ = (x1a_ - x1b_) / 2.6
wn_body = np.sqrt(2e4 * 2e5 / (2e4 + 2e5) / (600 * 1.4 / 2.6))
print(f"peak body heave {xc_.max() * 100:.2f} cm, peak pitch {np.degrees(np.abs(th_).max()):.3f} deg, peak front wheel {x2a_.max() * 100:.2f} cm (bump 5 cm)")
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, subplot_titles=("vertical displacement (cm)", "pitch angle (deg)"))
fig.add_scatter(x=ts[::3], y=np.round([road(x)[1] * 100 for x in ts[::3]], 3), name="road w_a", line=dict(dash="dot"), row=1, col=1)
fig.add_scatter(x=ts[::3], y=np.round(x2a_[::3] * 100, 3), name="front wheel x2a", row=1, col=1)
fig.add_scatter(x=ts[::3], y=np.round(xc_[::3] * 100, 3), name="body x_c", row=1, col=1)
fig.add_scatter(x=ts[::3], y=np.round(np.degrees(th_[::3]), 4), name="pitch", row=2, col=1)
fig.update_xaxes(title="t (s)", row=2, col=1)
show(fig, "Half-car passive suspension over a 5 cm bump (assumed parameters)", h=600)

peak body heave 0.57 cm, peak pitch 0.274 deg, peak front wheel 4.88 cm (bump 5 cm)
